# sequential-adapt: Colab quickstart

Runs the test suite, a smoke test, and the retention grid
(composed-state replay x hard orthogonal projection) on a Colab GPU.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

**Colab storage is ephemeral** — run the *Persist artifacts* cell at the
bottom before the VM disconnects, or your results are gone.

In [ ]:
# GPU sanity check
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
# Clone + install (torch is preinstalled on Colab)
import os
if not os.path.isdir("cc_fable_llm_lora_tests"):
    !git clone https://github.com/jmasseo/cc_fable_llm_lora_tests
%cd cc_fable_llm_lora_tests
!git pull
!pip install -q transformers

In [ ]:
# Test suite (~10 s, tiny random model; includes the new retention tests)
!python -m pytest tests -q

In [ ]:
# Smoke test: regression check that the existing pipeline still behaves
# (~1-2 min on GPU; downloads distilgpt2 on first run)
!python scripts/smoke_test.py

In [ ]:
# Single probe run of the new mechanisms before committing to the grid
!python scripts/run_controller.py --steps 200 --replay 1.0 --hard-ortho --no-gates \
    --out artifacts/controller_retention_probe.json

## Retention grid

2x2 grid: `{replay off/on} x {soft ortho / hard projection}`, all arms
`--no-gates`. Each `run_controller.py` call is ~70 s on an RTX 5070 Ti;
budget ~3-5x that on a T4.

- **Pilot** (next cell): 2 seeds -> 8 runs, roughly 30-45 min on a T4.
- **Full** (cell after): 10 seeds -> 40 runs, roughly 2.5-4 h on a T4.
  Consider the full run only on a paid tier or a faster runtime.

In [ ]:
# Pilot grid: 2 seeds
!SEEDS="0 1" REPLAY_VALUES="0 1.0" bash scripts/run_retention.sh

In [ ]:
# Full grid: 10 seeds (long!) — uncomment to run
# !bash scripts/run_retention.sh

In [ ]:
# Aggregate and render the summary inline
!python scripts/summarize_sweeps.py
from IPython.display import Markdown
Markdown(open("artifacts/sweeps_summary.md").read())

Reading the table: `retention_ctrl` is the baseline arm (no replay, soft
ortho), `replay=1` / `hard_ortho` / `replay=1;hard_ortho` are the treatment
arms. The question is whether *composed* controller retention (the known
failure) moves, and what it costs in current-task accuracy. See
`scripts/README.md` (Retention Grid section) for the comparison checklist.

## Persist artifacts (run before the VM disconnects)

In [ ]:
# Option A: zip + browser download
!zip -qr artifacts.zip artifacts
from google.colab import files
files.download("artifacts.zip")

In [ ]:
# Option B: copy to Google Drive — uncomment to use
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/sequential_adapt
# !cp -r artifacts /content/drive/MyDrive/sequential_adapt/